# 基于 Python 的药物数据分析与可视化

**说明：** 本 Notebook 使用 `data.csv` 中的教学模拟数据。它只用于练习 Pandas、Matplotlib 和 Jupyter，不能作为临床用药、药品定价或采购依据。

目标：读取数据、检查与清洗数据、进行描述性统计，并通过图表探索治疗领域、药物规格和模拟价格之间的关系。

## 1. 导入工具库

Pandas 用于表格数据处理，Matplotlib 用于绘图。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 让图表在 Notebook 内显示
%matplotlib inline

# 设置一个简洁的绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

## 2. 读取数据

数据文件和 Notebook 位于同一目录，因此直接使用相对路径 `data.csv`。

In [ ]:
file_path = 'data.csv'
drug_df = pd.read_csv(file_path)

print(f'原始数据共有 {drug_df.shape[0]} 行、{drug_df.shape[1]} 列。')
drug_df.head()

## 3. 数据检查

先查看每一列的数据类型、缺失值数量和重复情况。这是数据分析中很重要的第一步。

In [ ]:
# 查看数据类型和非空值数量
drug_df.info()

# 统计各列的缺失值
print('\n每列缺失值数量：')
print(drug_df.isna().sum())

# 以药物名称、治疗领域、规格和价格为依据检查重复记录
duplicate_columns = ['drug_name', 'therapeutic_area', 'dosage_mg', 'unit_price_cny']
duplicate_count = drug_df.duplicated(subset=duplicate_columns).sum()
print(f'\n按主要药物信息判断的重复记录数：{duplicate_count}')

## 4. 数据清洗

本数据特意保留了两个数值缺失值和一条重复药物记录，下面演示一种初学者常用的处理方式：

- 把规格和价格转换为数值；
- 使用该列的中位数补全缺失值，中位数受极端值影响较小；
- 删除主要药物信息完全相同的重复记录。

In [ ]:
# 使用副本，避免直接覆盖原始数据框
clean_df = drug_df.copy()

# 若读入时出现文本型数字，errors='coerce' 会把无法转换的值变为缺失值
numeric_columns = ['dosage_mg', 'unit_price_cny']
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors='coerce')
    median_value = clean_df[column].median()
    clean_df[column] = clean_df[column].fillna(median_value)
    print(f'{column} 的缺失值已用中位数 {median_value:.2f} 补全。')

# 删除重复的主要药物信息，只保留首次出现的记录
rows_before = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=duplicate_columns, keep='first').copy()
rows_after = len(clean_df)
print(f'删除了 {rows_before - rows_after} 条重复记录。')

# 再次确认清洗结果
print('\n清洗后各列缺失值数量：')
print(clean_df.isna().sum())
clean_df.head()

## 5. 描述性统计

`describe()` 可以快速给出数值列的计数、平均值、标准差、最小值和四分位数。之后再按治疗领域分组，比较各领域的药物数量与平均模拟价格。

In [ ]:
print('数值列的描述性统计：')
display(clean_df[['dosage_mg', 'unit_price_cny']].describe().round(2))

area_summary = (
    clean_df.groupby('therapeutic_area')
    .agg(
        drug_count=('drug_id', 'count'),
        average_price_cny=('unit_price_cny', 'mean'),
        median_price_cny=('unit_price_cny', 'median'),
        average_dosage_mg=('dosage_mg', 'mean')
    )
    .sort_values('average_price_cny', ascending=False)
    .round(2)
)

print('按治疗领域分组的统计结果：')
display(area_summary)

## 6. 可视化一：各治疗领域的平均模拟价格

柱状图适合比较不同类别的平均水平。

In [ ]:
plot_data = area_summary.sort_values('average_price_cny')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(plot_data.index, plot_data['average_price_cny'], color='#4C78A8')
ax.set_title('Average simulated price by therapeutic area', fontsize=14)
ax.set_xlabel('Average unit price (CNY per box)')
ax.set_ylabel('Therapeutic area')

# 在每个柱子旁写出数值，便于阅读
for bar in bars:
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height() / 2, f'{width:.1f}', va='center')

plt.tight_layout()
plt.show()

## 7. 可视化二：规格与模拟价格的关系

散点图用于观察两个数值变量是否有直观关系。颜色代表治疗领域；这里的点数较少，因此只能用于初步探索。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for area, group in clean_df.groupby('therapeutic_area'):
    ax.scatter(
        group['dosage_mg'],
        group['unit_price_cny'],
        label=area,
        s=70,
        alpha=0.8
    )

ax.set_title('Dosage and simulated unit price', fontsize=14)
ax.set_xlabel('Dosage (mg)')
ax.set_ylabel('Unit price (CNY per box)')
ax.legend(title='Therapeutic area', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. 可视化三：模拟价格的分布

直方图展示价格集中在哪些区间。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(clean_df['unit_price_cny'], bins=8, color='#F58518', edgecolor='white')
ax.axvline(
    clean_df['unit_price_cny'].mean(),
    color='#E45756',
    linestyle='--',
    label=f"Mean: {clean_df['unit_price_cny'].mean():.1f} CNY"
)
ax.set_title('Distribution of simulated unit price', fontsize=14)
ax.set_xlabel('Unit price (CNY per box)')
ax.set_ylabel('Number of drugs')
ax.legend()
plt.tight_layout()
plt.show()

## 9. 结果分析与局限性

从本模拟数据可以作出以下教学性观察：

1. 各治疗领域的平均模拟价格不同，抗病毒药、精神科药物等领域在本样本中相对较高。
2. 散点图没有显示出“规格越大、价格一定越高”的简单规律。药品价格还可能受剂型、包装数量、专利状态、企业类型和地区等因素影响。
3. 本例使用中位数补全少量缺失值，适合课堂练习；在真实研究中，应先追查缺失原因，并选择更严谨的处理方案。
4. 数据为人工生成的模拟数据，样本量很小，不能推广到真实药品市场或用于临床决策。

下一步可以尝试：增加真实且有明确授权的数据来源、按剂型或包装数量分层，或比较处方药与非处方药的价格分布。